## About this notebook:

This notebook uses Affinder napari plugin to find transformation manually.

Input:
- pathway to a directory containing segmented labels
- pathway to a directory containing data frames
- pathway to a directory containing transformations (pkl files)
- well and round (as in the names of directories) selected for corrections

Output:
- a set of transforms for the selected well with the corrected transform for a given round

Optionally you can visualize alignment (visualization uses Napari).

## Choose the experiment

Set `CONFIG` to the experiment's `experiment.json`. All folders and the well list come from it; create or edit it with `setup_experiment.ipynb`.

In [ ]:
from fouri import load_config

# This experiment's experiment.json (create or edit it with setup_experiment.ipynb)
CONFIG = r'Z:\PATH\TO\EXPERIMENT\experiment.json'

exp = load_config(CONFIG)

path_labels = exp.segmented   # segmented masks for alignment (output of 01)
path_df     = exp.output_df   # data frames (output of 00)
path_save   = exp.output_df   # transformations (output of 02) are read and updated here

# specify which round in which well to correct manually
myWell = exp.well_list[0]  # change to the well to correct, e.g. 'D03'
myRound = 0 # as named in the directory
anchor_round_selected = 1 # as named in the directory, default should be 1

## Prepare for processing

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import napari
from fouri import read_labels
from scipy import ndimage as ndi
from skimage import transform


In [ ]:
# open df
df_path = os.path.join(path_save, f'df_{myWell}.pkl')
df = pd.read_pickle(df_path)

# Print the unique values in the nameRound column to verify myRound is present
print("Unique values in 'nameRound':", df['nameRound'].unique())

# Convert myRound to the correct string format
myRound_str = f'R{myRound}'

# Verify that myRound is now in the correct format
print("Formatted 'myRound':", myRound_str)

# Check which alignRound for the selected round
filtered_align_round = df.loc[df.nameRound == myRound_str, 'alignRound']
print("Filtered 'alignRound' for 'myRound_str':", filtered_align_round)

align_round_list = filtered_align_round.tolist()

if align_round_list:
    alignRound = align_round_list[0]
    print(f"Selected alignRound: {alignRound}")
else:
    raise ValueError(f"No alignRound found for myRound = {myRound_str}. Please check the input data.")

# Check which alignRound for the anchor round
anchor_round_str = f'R{anchor_round_selected}'
anchor_round_list = df.loc[df.nameRound == anchor_round_str, 'alignRound'].tolist()

if anchor_round_list:
    anchor_round = anchor_round_list[0]
    print(f"Selected anchor_round: {anchor_round}")
else:
    raise ValueError(f"No alignRound found for anchor_round_selected = {anchor_round_str}. Please check the input data.")

# Open labels and the control
labels_im = read_labels(path_labels, myWell, indices=[anchor_round, alignRound])

# Open the list of transformations for a given well
tmat_path = os.path.join(path_save, f'tmat_{myWell}.pkl')
with open(tmat_path, 'rb') as tmat_file:
    tmat = pickle.load(tmat_file)

# Select the appropriate transformation
t_org = tmat[int(alignRound)]


## Use Affinder in Napari

Remember to set parameters of the plugin correctly:
- reference image
- image to transform
- 'similarity' as transformation

Information about Affinder can be found:
- https://github.com/jni/affinder
- https://www.napari-hub.org/plugins/affinder?imageModality=Fluorescence+microscopy

In [ ]:
viewer = napari.Viewer()
l0 = viewer.add_image(labels_im[0], colormap='bop blue', blending='additive',name='ref')
l1 = viewer.add_image(labels_im[1], colormap='bop purple', blending='additive',name='moving')

qtwidget, widget = viewer.window.add_plugin_dock_widget(
        'affinder', 'Start affinder'
        )
widget.reference.bind(l0)
widget.moving.bind(l1)
widget()

## Collect transformation matrix

In [ ]:
mat = np.asarray(viewer.layers['moving'].affine)

## Visualize transformation from the collected matrix (optional)

In [ ]:
# optionally - test matrix
tfd_ndi = ndi.affine_transform(labels_im[1], np.linalg.inv(mat))
viewer.add_image(tfd_ndi, colormap='bop orange', blending='additive')

## Save transformation matrix

In [ ]:
def matrix_rc2xy(affine_matrix):
    swapped_cols = affine_matrix[:, [1, 0, 2]]
    swapped_rows = swapped_cols[[1, 0, 2], :]
    return swapped_rows


tmat[int(alignRound),:,:] = np.linalg.inv(matrix_rc2xy(mat))
tmat.dump(tmat_path)